# Customer 360

Build a household-level analytical data product from the Silver layer by combining transaction behavior, campaign participation, coupon engagement, and demographic attributes.

**Silver Layer**  
↓  
Transaction Behavior  
↓  
Campaign Engagement  
↓  
Coupon Engagement  
↓  
Demographics  
↓  
**Customer 360 Gold Table**

### 1. Create Gold Schema

Create a dedicated Unity Catalog schema for business-ready analytical data products.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.consumer_analytics_gold;

In [0]:
SHOW SCHEMAS IN workspace;

databaseName
consumer_analytics_bronze
consumer_analytics_gold
consumer_analytics_silver
default
information_schema


### 2. Build Customer Transaction Metrics

Aggregate transaction-level Silver data to household level.

The resulting dataset uses **one row per household** and summarizes customer purchasing behavior including spend, baskets, product variety, activity frequency, and observed purchase period.

In [0]:
SELECT
    household_key,

    ROUND(SUM(SALES_VALUE), 2) AS total_spend,

    COUNT(DISTINCT BASKET_ID) AS total_baskets,

    COUNT(*) AS transaction_lines,

    COUNT(DISTINCT PRODUCT_ID) AS distinct_products,

    COUNT(DISTINCT DAY) AS active_days,

    MIN(DAY) AS first_purchase_day,

    MAX(DAY) AS last_purchase_day

FROM workspace.consumer_analytics_silver.transactions

GROUP BY household_key

ORDER BY total_spend DESC;

household_key,total_spend,total_baskets,transaction_lines,distinct_products,active_days,first_purchase_day,last_purchase_day
1023,38319.79,603,4403,1620,358,107,710
1609,27859.68,412,6625,1592,327,42,711
2322,23646.92,323,5692,2808,256,66,711
1453,21661.29,761,6561,3119,401,97,710
2459,20671.5,971,6646,3159,450,35,704
1430,20352.99,344,5372,1857,232,76,711
718,19299.86,599,6851,2844,375,1,707
707,19194.42,498,4310,1877,347,103,711
1653,19153.75,541,5347,2407,373,90,710
1111,18894.72,321,6576,1668,262,32,707


### 3. Validate Customer Transaction Grain

Validate that the aggregated transaction dataset contains exactly one record per household.

In [0]:
WITH customer_transactions AS (

    SELECT
        household_key,
        ROUND(SUM(SALES_VALUE), 2) AS total_spend,
        COUNT(DISTINCT BASKET_ID) AS total_baskets,
        COUNT(*) AS transaction_lines,
        COUNT(DISTINCT PRODUCT_ID) AS distinct_products,
        COUNT(DISTINCT DAY) AS active_days,
        MIN(DAY) AS first_purchase_day,
        MAX(DAY) AS last_purchase_day

    FROM workspace.consumer_analytics_silver.transactions

    GROUP BY household_key
)

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT household_key) AS distinct_households

FROM customer_transactions;

total_rows,distinct_households
2500,2500


### 4. Enrich Customer Transaction Metrics

Create additional household-level KPIs to describe customer purchasing behavior.

- `avg_basket_value`: average spend per shopping basket.
- `avg_spend_per_active_day`: average spend per active shopping day.

In [0]:
WITH customer_transactions AS (

    SELECT
        household_key,
        ROUND(SUM(SALES_VALUE), 2) AS total_spend,
        COUNT(DISTINCT BASKET_ID) AS total_baskets,
        COUNT(*) AS transaction_lines,
        COUNT(DISTINCT PRODUCT_ID) AS distinct_products,
        COUNT(DISTINCT DAY) AS active_days,
        MIN(DAY) AS first_purchase_day,
        MAX(DAY) AS last_purchase_day

    FROM workspace.consumer_analytics_silver.transactions

    GROUP BY household_key
)

SELECT
    *,
    ROUND(total_spend / total_baskets, 2) AS avg_basket_value,
    ROUND(total_spend / active_days, 2) AS avg_spend_per_active_day

FROM customer_transactions

ORDER BY total_spend DESC;

household_key,total_spend,total_baskets,transaction_lines,distinct_products,active_days,first_purchase_day,last_purchase_day,avg_basket_value,avg_spend_per_active_day
1023,38319.79,603,4403,1620,358,107,710,63.55,107.04
1609,27859.68,412,6625,1592,327,42,711,67.62,85.2
2322,23646.92,323,5692,2808,256,66,711,73.21,92.37
1453,21661.29,761,6561,3119,401,97,710,28.46,54.02
2459,20671.5,971,6646,3159,450,35,704,21.29,45.94
1430,20352.99,344,5372,1857,232,76,711,59.17,87.73
718,19299.86,599,6851,2844,375,1,707,32.22,51.47
707,19194.42,498,4310,1877,347,103,711,38.54,55.32
1653,19153.75,541,5347,2407,373,90,710,35.4,51.35
1111,18894.72,321,6576,1668,262,32,707,58.86,72.12


### 5. Build Campaign Engagement Metrics

Aggregate campaign participation to household level to measure how many distinct marketing campaigns each household was assigned to.

The resulting dataset maintains the Customer 360 grain of **one row per household**.

In [0]:
SELECT
    household_key,
    COUNT(DISTINCT CAMPAIGN) AS campaigns_assigned

FROM workspace.consumer_analytics_silver.campaign_households

GROUP BY household_key

ORDER BY campaigns_assigned DESC;

household_key,campaigns_assigned
2317,17
2489,16
1527,15
2459,15
718,15
676,14
1975,14
1917,14
982,14
979,13


### 6. Validate Campaign Metrics Grain

Validate that the campaign aggregation contains no more than one record per household.

In [0]:
WITH customer_campaigns AS (

    SELECT
        household_key,
        COUNT(DISTINCT CAMPAIGN) AS campaigns_assigned

    FROM workspace.consumer_analytics_silver.campaign_households

    GROUP BY household_key
)

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT household_key) AS distinct_households

FROM customer_campaigns;

total_rows,distinct_households
1584,1584


### 7. Build Coupon Redemption Metrics

Aggregate coupon redemption activity to household level.

The metrics capture the number of coupon redemption events, distinct coupons redeemed, and campaigns in which each household redeemed a coupon.

In [0]:
SELECT
    household_key,
    COUNT(*) AS coupon_redemption_events,
    COUNT(DISTINCT COUPON_UPC) AS distinct_coupons_redeemed,
    COUNT(DISTINCT CAMPAIGN) AS redemption_campaigns

FROM workspace.consumer_analytics_silver.coupon_redemptions

GROUP BY household_key

ORDER BY coupon_redemption_events DESC;

household_key,coupon_redemption_events,distinct_coupons_redeemed,redemption_campaigns
367,35,31,9
256,33,32,10
67,33,30,7
1823,30,27,6
931,29,29,3
979,28,26,8
1591,28,26,4
2489,28,27,6
1726,27,26,6
574,25,22,6


In [0]:
WITH customer_redemptions AS (

    SELECT
        household_key,
        COUNT(*) AS coupon_redemption_events,
        COUNT(DISTINCT COUPON_UPC) AS distinct_coupons_redeemed,
        COUNT(DISTINCT CAMPAIGN) AS redemption_campaigns

    FROM workspace.consumer_analytics_silver.coupon_redemptions

    GROUP BY household_key
)

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT household_key) AS distinct_households

FROM customer_redemptions;

total_rows,distinct_households
434,434


### 9. Build Customer 360

Combine household-level transaction behavior, campaign assignments, coupon redemptions, and demographic attributes into a single analytical dataset.

Transaction households form the base population. `LEFT JOIN` is used to preserve all purchasing households even when campaign, coupon, or demographic information is unavailable.

In [0]:
WITH customer_transactions AS (

    SELECT
        household_key,
        ROUND(SUM(SALES_VALUE), 2) AS total_spend,
        COUNT(DISTINCT BASKET_ID) AS total_baskets,
        COUNT(*) AS transaction_lines,
        COUNT(DISTINCT PRODUCT_ID) AS distinct_products,
        COUNT(DISTINCT DAY) AS active_days,
        MIN(DAY) AS first_purchase_day,
        MAX(DAY) AS last_purchase_day

    FROM workspace.consumer_analytics_silver.transactions

    GROUP BY household_key
),

customer_campaigns AS (

    SELECT
        household_key,
        COUNT(DISTINCT CAMPAIGN) AS campaigns_assigned

    FROM workspace.consumer_analytics_silver.campaign_households

    GROUP BY household_key
),

customer_redemptions AS (

    SELECT
        household_key,
        COUNT(*) AS coupon_redemption_events,
        COUNT(DISTINCT COUPON_UPC) AS distinct_coupons_redeemed,
        COUNT(DISTINCT CAMPAIGN) AS redemption_campaigns

    FROM workspace.consumer_analytics_silver.coupon_redemptions

    GROUP BY household_key
)

SELECT
    t.household_key,

    t.total_spend,
    t.total_baskets,
    ROUND(t.total_spend / t.total_baskets, 2) AS avg_basket_value,
    t.transaction_lines,
    t.distinct_products,
    t.active_days,
    t.first_purchase_day,
    t.last_purchase_day,

    COALESCE(c.campaigns_assigned, 0) AS campaigns_assigned,

    COALESCE(r.coupon_redemption_events, 0) AS coupon_redemption_events,
    COALESCE(r.distinct_coupons_redeemed, 0) AS distinct_coupons_redeemed,
    COALESCE(r.redemption_campaigns, 0) AS redemption_campaigns,

    d.AGE_DESC,
    d.MARITAL_STATUS_CODE,
    d.INCOME_DESC,
    d.HOMEOWNER_DESC,
    d.HH_COMP_DESC,
    d.HOUSEHOLD_SIZE_DESC,
    d.KID_CATEGORY_DESC

FROM customer_transactions t

LEFT JOIN customer_campaigns c
    ON t.household_key = c.household_key

LEFT JOIN customer_redemptions r
    ON t.household_key = r.household_key

LEFT JOIN workspace.consumer_analytics_silver.demographics d
    ON t.household_key = d.household_key;

household_key,total_spend,total_baskets,avg_basket_value,transaction_lines,distinct_products,active_days,first_purchase_day,last_purchase_day,campaigns_assigned,coupon_redemption_events,distinct_coupons_redeemed,redemption_campaigns,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC
1733,1487.37,34,43.75,437,325,31,8,709,2,0,0,0,null,null,null,null,null,null,null
586,5822.64,211,27.6,1975,1063,175,52,711,8,0,0,0,19-24,A,35-49K,Renter,2 Adults Kids,3,1
1720,5755.69,142,40.53,1847,842,139,82,704,8,0,0,0,45-54,U,50-74K,Unknown,Unknown,1,None/Unknown
1303,6344.09,212,29.92,1654,1165,170,98,708,5,1,1,1,null,null,null,null,null,null,null
1798,2457.51,42,58.51,964,640,37,98,700,1,0,0,0,null,null,null,null,null,null,null
1146,8854.64,345,25.67,2855,1029,249,96,711,9,12,10,4,35-44,B,35-49K,Homeowner,1 Adult Kids,4,3+
36,4898.25,221,22.16,1359,681,182,88,708,3,0,0,0,null,null,null,null,null,null,null
97,5323.95,180,29.58,1818,968,164,43,706,4,0,0,0,45-54,U,75-99K,Unknown,Single Female,1,None/Unknown
384,3914.84,80,48.94,1075,777,69,105,706,5,0,0,0,null,null,null,null,null,null,null
282,4162.56,110,37.84,1259,722,93,61,711,5,6,6,2,45-54,U,35-49K,Unknown,Single Female,1,None/Unknown


### 10. Validate Customer 360 Grain

Validate that the combined Customer 360 dataset preserves the intended grain of **one row per household** after all joins.

In [0]:
WITH customer_transactions AS (

    SELECT
        household_key,
        ROUND(SUM(SALES_VALUE), 2) AS total_spend,
        COUNT(DISTINCT BASKET_ID) AS total_baskets,
        COUNT(*) AS transaction_lines,
        COUNT(DISTINCT PRODUCT_ID) AS distinct_products,
        COUNT(DISTINCT DAY) AS active_days,
        MIN(DAY) AS first_purchase_day,
        MAX(DAY) AS last_purchase_day

    FROM workspace.consumer_analytics_silver.transactions
    GROUP BY household_key
),

customer_campaigns AS (

    SELECT
        household_key,
        COUNT(DISTINCT CAMPAIGN) AS campaigns_assigned

    FROM workspace.consumer_analytics_silver.campaign_households
    GROUP BY household_key
),

customer_redemptions AS (

    SELECT
        household_key,
        COUNT(*) AS coupon_redemption_events,
        COUNT(DISTINCT COUPON_UPC) AS distinct_coupons_redeemed,
        COUNT(DISTINCT CAMPAIGN) AS redemption_campaigns

    FROM workspace.consumer_analytics_silver.coupon_redemptions
    GROUP BY household_key
),

customer_360 AS (

    SELECT
        t.household_key

    FROM customer_transactions t

    LEFT JOIN customer_campaigns c
        ON t.household_key = c.household_key

    LEFT JOIN customer_redemptions r
        ON t.household_key = r.household_key

    LEFT JOIN workspace.consumer_analytics_silver.demographics d
        ON t.household_key = d.household_key
)

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT household_key) AS distinct_households

FROM customer_360;

total_rows,distinct_households
2500,2500


### 11. Persist Customer 360 Gold Table

Persist the validated household-level Customer 360 dataset as a managed Delta table in the Gold layer.

The table provides a reusable analytical data product combining purchasing behavior, campaign assignments, coupon redemption activity, and demographic attributes.

In [0]:
CREATE OR REPLACE TABLE workspace.consumer_analytics_gold.customer_360
USING DELTA
AS

WITH customer_transactions AS (

    SELECT
        household_key,
        ROUND(SUM(SALES_VALUE), 2) AS total_spend,
        COUNT(DISTINCT BASKET_ID) AS total_baskets,
        COUNT(*) AS transaction_lines,
        COUNT(DISTINCT PRODUCT_ID) AS distinct_products,
        COUNT(DISTINCT DAY) AS active_days,
        MIN(DAY) AS first_purchase_day,
        MAX(DAY) AS last_purchase_day

    FROM workspace.consumer_analytics_silver.transactions
    GROUP BY household_key
),

customer_campaigns AS (

    SELECT
        household_key,
        COUNT(DISTINCT CAMPAIGN) AS campaigns_assigned

    FROM workspace.consumer_analytics_silver.campaign_households
    GROUP BY household_key
),

customer_redemptions AS (

    SELECT
        household_key,
        COUNT(*) AS coupon_redemption_events,
        COUNT(DISTINCT COUPON_UPC) AS distinct_coupons_redeemed,
        COUNT(DISTINCT CAMPAIGN) AS redemption_campaigns

    FROM workspace.consumer_analytics_silver.coupon_redemptions
    GROUP BY household_key
)

SELECT
    t.household_key,

    t.total_spend,
    t.total_baskets,
    ROUND(t.total_spend / t.total_baskets, 2) AS avg_basket_value,
    t.transaction_lines,
    t.distinct_products,
    t.active_days,
    t.first_purchase_day,
    t.last_purchase_day,

    COALESCE(c.campaigns_assigned, 0) AS campaigns_assigned,

    COALESCE(r.coupon_redemption_events, 0) AS coupon_redemption_events,
    COALESCE(r.distinct_coupons_redeemed, 0) AS distinct_coupons_redeemed,
    COALESCE(r.redemption_campaigns, 0) AS redemption_campaigns,

    d.AGE_DESC,
    d.MARITAL_STATUS_CODE,
    d.INCOME_DESC,
    d.HOMEOWNER_DESC,
    d.HH_COMP_DESC,
    d.HOUSEHOLD_SIZE_DESC,
    d.KID_CATEGORY_DESC,

    CURRENT_TIMESTAMP() AS _created_at

FROM customer_transactions t

LEFT JOIN customer_campaigns c
    ON t.household_key = c.household_key

LEFT JOIN customer_redemptions r
    ON t.household_key = r.household_key

LEFT JOIN workspace.consumer_analytics_silver.demographics d
    ON t.household_key = d.household_key;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM workspace.consumer_analytics_gold.customer_360
LIMIT 10

household_key,total_spend,total_baskets,avg_basket_value,transaction_lines,distinct_products,active_days,first_purchase_day,last_purchase_day,campaigns_assigned,coupon_redemption_events,distinct_coupons_redeemed,redemption_campaigns,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,_created_at
1733,1487.37,34,43.75,437,325,31,8,709,2,0,0,0,null,null,null,null,null,null,null,2026-09-24T06:11:15.047Z
586,5822.64,211,27.6,1975,1063,175,52,711,8,0,0,0,19-24,A,35-49K,Renter,2 Adults Kids,3,1,2026-09-24T06:11:15.047Z
1720,5755.69,142,40.53,1847,842,139,82,704,8,0,0,0,45-54,U,50-74K,Unknown,Unknown,1,None/Unknown,2026-09-24T06:11:15.047Z
1303,6344.09,212,29.92,1654,1165,170,98,708,5,1,1,1,null,null,null,null,null,null,null,2026-09-24T06:11:15.047Z
1798,2457.51,42,58.51,964,640,37,98,700,1,0,0,0,null,null,null,null,null,null,null,2026-09-24T06:11:15.047Z
1146,8854.64,345,25.67,2855,1029,249,96,711,9,12,10,4,35-44,B,35-49K,Homeowner,1 Adult Kids,4,3+,2026-09-24T06:11:15.047Z
36,4898.25,221,22.16,1359,681,182,88,708,3,0,0,0,null,null,null,null,null,null,null,2026-09-24T06:11:15.047Z
97,5323.95,180,29.58,1818,968,164,43,706,4,0,0,0,45-54,U,75-99K,Unknown,Single Female,1,None/Unknown,2026-09-24T06:11:15.047Z
384,3914.84,80,48.94,1075,777,69,105,706,5,0,0,0,null,null,null,null,null,null,null,2026-09-24T06:11:15.047Z
282,4162.56,110,37.84,1259,722,93,61,711,5,6,6,2,45-54,U,35-49K,Unknown,Single Female,1,None/Unknown,2026-09-24T06:11:15.047Z


### 12. Validate Persisted Gold Table

Perform final validation checks on the persisted Customer 360 Gold table.

The validation confirms:
- one row per household,
- expected customer population,
- demographic coverage,
- campaign and coupon metrics are stored correctly.

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT household_key) AS distinct_households,

    COUNT(CASE WHEN AGE_DESC IS NOT NULL THEN 1 END) AS households_with_demographics,

    COUNT(CASE WHEN campaigns_assigned > 0 THEN 1 END) AS households_with_campaigns,

    COUNT(CASE WHEN coupon_redemption_events > 0 THEN 1 END) AS households_with_redemptions

FROM workspace.consumer_analytics_gold.customer_360;

total_rows,distinct_households,households_with_demographics,households_with_campaigns,households_with_redemptions
2500,2500,801,1584,434


### 13. Inspect Customer 360

Review a sample of the final Gold dataset to confirm that behavioral, campaign, coupon, and demographic attributes are represented as expected.

In [0]:
SELECT
    household_key,
    total_spend,
    total_baskets,
    avg_basket_value,
    distinct_products,
    active_days,
    campaigns_assigned,
    coupon_redemption_events,
    distinct_coupons_redeemed,
    AGE_DESC,
    INCOME_DESC,
    HOUSEHOLD_SIZE_DESC

FROM workspace.consumer_analytics_gold.customer_360

ORDER BY total_spend DESC

LIMIT 20;

household_key,total_spend,total_baskets,avg_basket_value,distinct_products,active_days,campaigns_assigned,coupon_redemption_events,distinct_coupons_redeemed,AGE_DESC,INCOME_DESC,HOUSEHOLD_SIZE_DESC
1023,38319.79,603,63.55,1620,358,0,0,0,null,null,null
1609,27859.68,412,67.62,1592,327,3,4,4,45-54,125-149K,5+
2322,23646.92,323,73.21,2808,256,10,3,3,45-54,175-199K,1
1453,21661.29,761,28.46,3119,401,13,21,19,45-54,125-149K,3
2459,20671.5,971,21.29,3159,450,15,0,0,null,null,null
1430,20352.99,344,59.17,1857,232,11,1,1,35-44,35-49K,3
718,19299.86,599,32.22,2844,375,15,5,5,45-54,25-34K,5+
707,19194.42,498,38.54,1877,347,9,0,0,25-34,100-124K,5+
1653,19153.75,541,35.4,2407,373,10,1,1,35-44,Under 15K,1
1111,18894.72,321,58.86,1668,262,12,0,0,null,null,null
